In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image


In [3]:
transform = transforms.Compose([
    transforms.Resize((64,64)),            # Resize images to 64x64
    transforms.Grayscale(num_output_channels=1),  # Convert to 1 channel
    transforms.ToTensor(),                 # Convert to tensor
    transforms.Normalize([0.5],[0.5])     # Normalize pixel values
])


In [4]:
train_dataset = datasets.ImageFolder('dataset/train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


In [5]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,16,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(16,32,3,padding=1)
        self.fc1 = nn.Linear(32*16*16, 2)  # 2 output classes

    def forward(self,x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 32*16*16)  # flatten
        x = self.fc1(x)
        return x

model = SimpleCNN()  # initialize model


In [6]:
criterion = nn.CrossEntropyLoss()  # suitable for classification
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [7]:
# num_epochs = 5  # train for 5 passes over the dataset

# for epoch in range(num_epochs):
#     model.train()  # set model to training mode

#     for images, labels in train_loader:
#         optimizer.zero_grad()
#         outputs = model(images)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()

#=================================#
num_epochs = 5  # train for 5 times over the dataset

for epoch in range(num_epochs):
    model.train()  # set model to training mode
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {100*correct/total:.2f}%")



Epoch 1, Loss: 0.2645, Accuracy: 88.61%
Epoch 2, Loss: 0.1263, Accuracy: 95.32%
Epoch 3, Loss: 0.1023, Accuracy: 96.47%
Epoch 4, Loss: 0.0876, Accuracy: 97.11%
Epoch 5, Loss: 0.0712, Accuracy: 97.36%


In [8]:
torch.save(model.state_dict(), "xray_cnn.pth")
print("Model saved as xray_cnn.pth")


Model saved as xray_cnn.pth


In [9]:
model = SimpleCNN()  # define the model again
model.load_state_dict(torch.load("xray_cnn.pth"))
model.eval()  # set model to evaluation mode
print("Model loaded and ready for prediction")


Model loaded and ready for prediction


In [10]:
def predict_xray(image_path):
    img = Image.open(image_path).convert('L')  # grayscale
    transform = transforms.Compose([
        transforms.Resize((64,64)),
        transforms.ToTensor(),
        transforms.Normalize([0.5],[0.5])
    ])
    img = transform(img)
    img = img.unsqueeze(0)  # add batch dimension
    output = model(img)
    _, predicted = torch.max(output, 1)
    classes = ['Normal', 'Pneumonia']
    print("Prediction:", classes[predicted.item()])

# Example usage:
#predict_xray('your_image.jpg')  # replace with your image path
# predict_xray('dataset/train/Viral Pneumonia/5.jpg')
# predict_xray('dataset/train/normal/5.jpg')
predict_xray('dataset/train/Viral Pneumonia/9.jpg')


Prediction: Pneumonia


In [14]:
# Make sure the model is in evaluation mode
model.eval()  

correct = 0
total = 0

# We don't need gradients for calculating accuracy
with torch.no_grad():
    for images, labels in train_loader:  # use the training data
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

train_accuracy = 100 * correct / total
print(f"Training Accuracy: {train_accuracy:.2f}%")


Training Accuracy: 97.61%


In [15]:
import os
from PIL import Image
import torch
from torchvision import transforms

# Make sure your model is loaded
# model = SimpleCNN()
# model.load_state_dict(torch.load("xray_cnn.pth"))
# model.eval()

# Classes
classes = ['Normal', 'Pneumonia']

# Transform (same as training)
transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize([0.5],[0.5])
])

def predict_folder(folder_path):
    """
    Predict all images in a folder.
    If the folder contains subfolders (like train/test structure), 
    it uses the subfolder name as true label.
    """
    # Check if folder has subfolders
    subfolders = [f for f in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path,f))]

    if subfolders:  # structured folder with labels
        for sub in subfolders:
            sub_path = os.path.join(folder_path, sub)
            for file_name in os.listdir(sub_path):
                if file_name.endswith('.jpg') or file_name.endswith('.png'):
                    image_path = os.path.join(sub_path, file_name)
                    img = Image.open(image_path).convert('L')
                    img = transform(img)
                    img = img.unsqueeze(0)
                    output = model(img)
                    _, predicted = torch.max(output, 1)
                    print(f"{file_name} | True: {sub} | Predicted: {classes[predicted.item()]}")
    else:  # folder with just images, no subfolders
        for file_name in os.listdir(folder_path):
            if file_name.endswith('.jpg') or file_name.endswith('.png'):
                image_path = os.path.join(folder_path, file_name)
                img = Image.open(image_path).convert('L')
                img = transform(img)
                img = img.unsqueeze(0)
                output = model(img)
                _, predicted = torch.max(output, 1)
                print(f"{file_name} | Predicted: {classes[predicted.item()]}")


# ===============================
# Example usage:
# ===============================
# Predict all images in the test folder
predict_xray('dataset/train/Normal/14.jpg')

# Predict all images in a downloaded folder
# predict_folder('new_xray_images')


Prediction: Normal
